# DIP-STER Reconstruction notebook
In this notebook, we reconstruct a volume-time series from a trained DIP-STER model.


## 1. Imports and Initialization

This section performs all the necessary imports and preparations for the model to reconstruct the tilt series.

In [1]:
## Standard library imports
import os
import shutil
import logging
## Data Processing Imports
import numpy as np
import torch
import gc

## Image Processing Imports
import tomobase
tomobase.core.logger.setLevel(logging.INFO)

from tomobase.core.data_classes import Sinogram, Volume
import dipster as dip

## Plotting Imports
import matplotlib.pyplot as plt
import stackview

tomobase.bootstrap(qt_enabled=True)

2026-04-01 16:59:44,511 - INFO - CuPy is available. 1 GPU devices detected.
2026-04-01 16:59:44,560 - DEBUG - Added category 'Acquistion' -> 0x3c00000000 (inheritor=None)
2026-04-01 16:59:44,560 - DEBUG - Added category 'Tomography' -> 0x4400000000 (inheritor=None)
2026-04-01 16:59:44,561 - DEBUG - Added category 'Visualization' -> 0x4c00000000 (inheritor=None)
2026-04-01 16:59:44,561 - DEBUG - Added category 'Deform' -> 0x443c000000 (inheritor=Tomography)
2026-04-01 16:59:44,561 - DEBUG - Added category 'Image Processing' -> 0x4440000000 (inheritor=Tomography)
2026-04-01 16:59:44,561 - DEBUG - Added category 'Align' -> 0x4444000000 (inheritor=Tomography)
2026-04-01 16:59:44,562 - DEBUG - Added category 'Reconstruct' -> 0x4448000000 (inheritor=Tomography)
2026-04-01 16:59:44,562 - DEBUG - Added category 'Project' -> 0x444c000000 (inheritor=Tomography)
2026-04-01 16:59:44,562 - DEBUG - Added category 'Segment' -> 0x4450000000 (inheritor=Tomography)
2026-04-01 16:59:44,562 - DEBUG - Adde

Loading plugins...
Loading plugins...


In [ ]:
#### User Input ####

## Select the root data folder containing all subsequent directories
main_dir =  '/home/amoncomble/Programs/~DIPSTERv2/newdata' #'/path/to/main/directory'

tiltseries_dir = 'TiltSeries128'  # Tilt Series folder 
model_dir = 'Model128'  # Trained DIP-STER Models folder
reconstruction_dir = 'DIP128'  # DIP-STER Reconstruction folder

tiltseries_file = 'Simu_Nanoshell.mrc'  # File containing the tilt series
model_file = 'Exp_AuAgCube_twilight-shape.dip.pkl' #'Simu_Nanoshell_sage-forest.dip.pkl'  # File containing the model

#### - - - - - ####

## Definition of all the necessary paths
path_tiltseries = os.path.join(main_dir, tiltseries_dir, tiltseries_file)
path_model = os.path.join(main_dir, model_dir, model_file)
path_reconstruction = os.path.join(main_dir, reconstruction_dir, model_file.split('.')[0])

## 2. Reconstruction

In DIP-STER, we infer orthoslices by the defining manifold coordinates $\left(y,t,\theta\right)$. In the case of the reconstruction of a volume, we want all 128 orthoslices along $y$. The volume-time series correspond to the reconstruction of N volumes with associated $t$ and $\theta$.

### 2.1 Choice of the time steps and angles

By default a reconstruction is made with the tilt series time steps and angles, but user can define the $t$ and $\theta$ wanted.

In [ ]:
#### Loading tilt series ####

tiltseries = Sinogram.read(path_tiltseries)

####  Pre visualization  ####
## Rotation axis must be vertical

tiltseries.data.transpose('projections','y','x')
tiltseries.sort(by='times')  # Change it to 'angles' to have a better view of the rotation axis
stackview.slice(tiltseries.data.values)

In [ ]:
tiltseries.sort(by='times')  # Reset ordering

#### Tilt Series time and angle  ####

reconstruction_times = tiltseries.times
reconstruction_angles = tiltseries.angles

#### - - - - - - - - - - - - - - ####

## OR

#### User defined time and angle ####

# reconstruction_times = np.arange(0, 50)
# reconstruction_angles = np.zeros_like(tiltseries.times)

#### - - - - - - - - - - - - - - ####

print('Time Steps: ', reconstruction_times)
print('Angles: ', reconstruction_angles)

In [ ]:
#### Converting data into Pytorch ####

reconstruction_angles =torch.from_numpy(reconstruction_angles).to("cuda")
reconstruction_times = torch.from_numpy(reconstruction_times).to("cuda")

### 2.2 Model -> Volume-Series folder (REC/MRC)

To analyze the reconstructed volumes, we mostly worked with AMIRA that uses .rec files but they are equivalent to .mrc files. They can easily be open and analysed with 3D plugins in Fiji/ImageJ.

In [ ]:
####   Loading the trained model   ####

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()
net = dip.Solver.from_state_dict(torch.load(path_model))

In [ ]:
####  Create reconstruction folder ####

if os.path.exists(path_reconstruction):
    shutil.rmtree(path_reconstruction)
os.mkdir(path_reconstruction)

In [ ]:
#### Reconstruction of the volumes ####

proj_size = net.params.proj_size

for i in range(len(reconstruction_times)):
    rec = np.zeros((proj_size, proj_size, proj_size))  # 128 x 128 x 128 volume
    tiled_angles = torch.full((proj_size,), reconstruction_angles[i]).to(net.params.dev)
    tiled_times = torch.full((proj_size,), reconstruction_times[i]).to(net.params.dev)
    rec[:,:,:] = dip.util.torch_to_np(net.reconstruct_slices(tiled_angles, np.arange(proj_size), tiled_times)).squeeze()

    volume_reconstructed = Volume(f'DIP_rec_{i}', rec, dict={'times':reconstruction_times[i], 'angles':reconstruction_angles[i], 'model':path_model.split('/')[-1]})
    volume_reconstructed.write(os.path.join(path_reconstruction, f'{i}_data.rec'))